In [1]:
import pandas as pd
import numpy as np
from urllib.parse import quote

import json
import re
from tqdm.auto import tqdm

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

/data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset
edge_data = pd.read_csv('../../data/kc_dataset/tutorialbank/prerequisite_annotations.csv')
kc_data = pd.read_csv('../../data/kc_dataset/tutorialbank/prerequisite_topics.csv')
kc_data.columns = kc_data.columns.str.strip()
print(kc_data.info())
kc_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   topic_id        208 non-null    int64 
 1   topic_name      208 non-null    object
 2   wikipedia_link  184 non-null    object
dtypes: int64(1), object(2)
memory usage: 5.0+ KB
None


,topic_id,topic_name,wikipedia_link
0,2,Anaphora Resolution,https://en.wikipedia.org/wiki/Anaphora_(lingui...
1,3,Generative Adversarial Networks,https://en.wikipedia.org/wiki/Generative_adver...
2,4,Neural Machine Translation,https://en.wikipedia.org/wiki/Neural_machine_t...
3,5,Variational Autoencoders,https://en.wikipedia.org/wiki/Autoencoder#Vari...
4,6,IBM Translation Models,https://en.wikipedia.org/wiki/IBM_alignment_mo...


In [3]:
print(edge_data.info())
edge_data.columns = edge_data.columns.str.strip()
edge_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42643 entries, 0 to 42642
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   source_topic_id   42643 non-null  int64
 1    target_topic_id  42643 non-null  int64
 2    prereq_relation  42643 non-null  int64
 3   annotator_id      42643 non-null  int64
dtypes: int64(4)
memory usage: 1.3 MB
None


,source_topic_id,target_topic_id,prereq_relation,annotator_id
0,2,3,-1,56
1,2,4,-1,3
2,2,5,-1,3
3,2,6,-1,3
4,2,7,-1,15


### 전처리

In [4]:
edge_data[edge_data["prereq_relation"] == 1].shape

(904, 4)

In [5]:
## prereq_relation이 1이 아니면 삭제

edge_data = edge_data[edge_data["prereq_relation"] == 1]
edge_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 904 entries, 34 to 42221
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   source_topic_id  904 non-null    int64
 1   target_topic_id  904 non-null    int64
 2   prereq_relation  904 non-null    int64
 3   annotator_id     904 non-null    int64
dtypes: int64(4)
memory usage: 35.3 KB


In [6]:
# 결측치 처리
# kc_data 의 wikipedia_link에서 none인 것들만 찾기
link_none_df = kc_data[kc_data['wikipedia_link'].isna()]
link_none_df.info()

## 삭제
## 만약 해당 topic id가 edge_data의 source_topic_id, target_topic_id에 있다면 엣지도 삭제해야 함

ids_to_remove = set(link_none_df['topic_id'].tolist())
edge_data = edge_data[~edge_data['source_topic_id'].isin(ids_to_remove)]
edge_data = edge_data[~edge_data['target_topic_id'].isin(ids_to_remove)]
kc_data = kc_data[~kc_data["topic_id"].isin(ids_to_remove)]
edge_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24 entries, 7 to 207
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   topic_id        24 non-null     int64 
 1   topic_name      24 non-null     object
 2   wikipedia_link  0 non-null      object
dtypes: int64(1), object(2)
memory usage: 768.0+ bytes
<class 'pandas.core.frame.DataFrame'>
Index: 577 entries, 48 to 40319
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   source_topic_id  577 non-null    int64
 1   target_topic_id  577 non-null    int64
 2   prereq_relation  577 non-null    int64
 3   annotator_id     577 non-null    int64
dtypes: int64(4)
memory usage: 22.5 KB


### node, edge 1차 데이터셋 생성

In [7]:
## kc_data 내 wikipedia_link에서 name 뽑아내서 새로운 칼럼으로 추가
## https://en.wikipedia.org/wiki/Machine_learning  -> Machine_learning

## nodes 라는 변수에다가 df 할당해두고
## 칼럼은 id, name, tb_name (= topic_name), link (= wikipedia_link), alias (지금은 다 none)

kc_data = kc_data.copy()

# name 추출: https://en.wikipedia.org/wiki/Machine_learning -> Machine_learning
# - 뒤에 ?query, #anchor 붙는 경우도 제거
kc_data["name"] = (
    kc_data["wikipedia_link"]
      .astype("string")
      .str.replace(r"[?#].*$", "", regex=True)   # ?... 또는 #... 제거
      .str.rstrip("/")                          # 끝에 / 제거
      .str.split("/")
      .str[-1]
)

# nodes 생성:
nodes = pd.DataFrame({
    "id": kc_data["topic_id"],
    "name": kc_data["name"],
    "tb_name": kc_data["topic_name"],
    "link": kc_data["wikipedia_link"],
    "alias": pd.NA,  # 전부 None
})


nodes.info()
nodes.head()

<class 'pandas.core.frame.DataFrame'>
Index: 184 entries, 0 to 199
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       184 non-null    int64 
 1   name     184 non-null    object
 2   tb_name  184 non-null    object
 3   link     184 non-null    object
 4   alias    0 non-null      object
dtypes: int64(1), object(4)
memory usage: 8.6+ KB


,id,name,tb_name,link,alias
0,2,Anaphora_(linguistics),Anaphora Resolution,https://en.wikipedia.org/wiki/Anaphora_(lingui...,<NA>
1,3,Generative_adversarial_network,Generative Adversarial Networks,https://en.wikipedia.org/wiki/Generative_adver...,<NA>
2,4,Neural_machine_translation,Neural Machine Translation,https://en.wikipedia.org/wiki/Neural_machine_t...,<NA>
3,5,Autoencoder,Variational Autoencoders,https://en.wikipedia.org/wiki/Autoencoder#Vari...,<NA>
4,6,IBM_alignment_models,IBM Translation Models,https://en.wikipedia.org/wiki/IBM_alignment_mo...,<NA>


In [8]:
## source_topic_id: 선수개념
## target_topic_id: 이해하고자하는 개념
## source_topic_id (= prereq_name) -> target_topic_id (= name)

## edges 라는 데이터프레임 생성
## 칼럼: name (= target_topic_id), prereq_name (= source_topic_id), strength, reason
## strength, reason는 일단 pd.NA

edge_data = edge_data.copy()

nodes = nodes.copy()
nodes.columns = nodes.columns.str.strip()

# id -> name 매핑 dict 만들기
id_to_name = nodes.set_index("id")["name"].to_dict()

# source/target id를 name으로 매핑
edge_data["prereq_name"] = edge_data["source_topic_id"].map(id_to_name)
edge_data["name"] = edge_data["target_topic_id"].map(id_to_name)

# 매핑 안 된 row 확인 (있으면 nodes에 없는 id가 edge에 있다는 뜻)
missing = edge_data[edge_data["prereq_name"].isna() | edge_data["name"].isna()]
print("매핑 누락 row 수:", len(missing))

# edges DF 생성 (원하는 컬럼만)
edges = edge_data[["name", "prereq_name"]].copy()
edges["strength"] = pd.NA
edges["reason"] = pd.NA

edges.info()
edges.head()

매핑 누락 row 수: 0
<class 'pandas.core.frame.DataFrame'>
Index: 577 entries, 48 to 40319
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         577 non-null    object
 1   prereq_name  577 non-null    object
 2   strength     0 non-null      object
 3   reason       0 non-null      object
dtypes: object(4)
memory usage: 22.5+ KB


,name,prereq_name,strength,reason
48,Word-sense_disambiguation,Anaphora_(linguistics),<NA>,<NA>
68,Dialog_system,Anaphora_(linguistics),<NA>,<NA>
207,Autoencoder,Generative_adversarial_network,<NA>,<NA>
436,Semantic_analysis_(computational),Neural_machine_translation,<NA>,<NA>
558,Statistical_machine_translation,Neural_machine_translation,<NA>,<NA>


### strength, reason, alias 생성
- 사용 모델: JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4

In [9]:
MODEL = "JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4"  ## "Qwen/Qwen3-30B-A3B-GPTQ-Int4"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

llm = LLM(
    model=MODEL,
    trust_remote_code=True,
    max_model_len=2048,
    gpu_memory_utilization=0.9,
    tensor_parallel_size=1,
    enable_prefix_caching=True,
    enforce_eager=True,
)

sampling = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=256,
)

def extract_json(text: str):
    if text is None:
        return None
    t = text.strip()
    if not t:
        return None

    # <think>...</think> 제거
    t = re.sub(r"<think>[\s\S]*?</think>", "", t).strip()

    # 코드블록(JSON) 우선 시도
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", t, flags=re.IGNORECASE)
    if m:
        cand = m.group(1).strip()
        # (a) 그대로
        try:
            return json.loads(cand)
        except Exception:
            # (b) 흔한 누락 보정: 마지막 } 없으면 붙여보기
            c2 = cand
            if c2.startswith("{") and not c2.endswith("}"):
                try:
                    return json.loads(c2 + "}")
                except Exception:
                    pass

    # 텍스트에서 JSON 객체 구간 추출
    start = t.find("{")
    if start == -1:
        return None

    # 마지막 }가 있으면 그 구간으로 파싱, 없으면 끝까지를 후보로
    end = t.rfind("}")
    if end != -1 and end > start:
        cand = t[start:end+1].strip()
        try:
            return json.loads(cand)
        except Exception:
            # 마지막 }가 있었는데도 실패하면, 혹시 뒤에 군더더기/잘림 대비 보정도 시도
            pass
    else:
        cand = t[start:].strip()

    # (a) 그대로
    try:
        return json.loads(cand)
    except Exception:
        pass

    # (b) 마지막 } 누락 보정
    if cand.startswith("{") and not cand.endswith("}"):
        try:
            return json.loads(cand + "}")
        except Exception:
            pass

    # (c) 마지막 ]까진 있는데 }만 없는 형태 보정: ...]  -> ...]}
    if cand.startswith("{") and not cand.endswith("}") and cand.rstrip().endswith("]"):
        try:
            return json.loads(cand.rstrip() + "}")
        except Exception:
            pass

    return None


def build_chat_prompt(system: str, user: str) -> str:
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

INFO 01-25 16:50:13 [utils.py:263] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'enable_prefix_caching': True, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 01-25 16:50:15 [model.py:530] Resolved architecture: Qwen3MoeForCausalLM
WARNING 01-25 16:50:15 [model.py:1817] Your device 'Tesla V100-SXM2-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 01-25 16:50:15 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 01-25 16:50:15 [model.py:1545] Using max model len 2048


2026-01-25 16:50:17,500	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-25 16:50:17 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-25 16:50:17 [gptq.py:99] Currently, the 4-bit gptq_gemm kernel for GPTQ is buggy. Please switch to gptq_marlin or gptq_bitblas.


...

INFO 01-25 16:50:26 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-25 16:50:26 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
WARNING 01-25 16:50:26 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 01-25 16:50:26 [vllm.py:765] Cudagraph is disabled under eager mode
WARNING 01-25 16:50:28 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=40254) INFO 01-25 16:50:36 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', speculative_config=None, tokenizer='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch

(EngineCore_DP0 pid=40254) /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=40254) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=40254)   warnings.warn(


(EngineCore_DP0 pid=40254) INFO 01-25 16:50:41 [cuda.py:351] Using TRITON_ATTN attention backend out of potential backends: ('TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:03<00:15,  3.98s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:04<00:06,  2.09s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:09<00:06,  3.16s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:13<00:03,  3.67s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:18<00:00,  3.95s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:18<00:00,  3.61s/it]
(EngineCore_DP0 pid=40254) 


(EngineCore_DP0 pid=40254) INFO 01-25 16:51:07 [default_loader.py:291] Loading weights took 18.17 seconds
(EngineCore_DP0 pid=40254) INFO 01-25 16:51:08 [gpu_model_runner.py:3905] Model loading took 15.59 GiB memory and 29.661408 seconds
(EngineCore_DP0 pid=40254) WARNING 01-25 16:51:09 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128,N=768,device_name=Tesla_V100-SXM2-32GB,dtype=int4_w4a16.json
(EngineCore_DP0 pid=40254) INFO 01-25 16:51:27 [gpu_worker.py:358] Available KV cache memory: 11.53 GiB
(EngineCore_DP0 pid=40254) INFO 01-25 16:51:27 [kv_cache_utils.py:1305] GPU KV cache size: 125,888 tokens
(EngineCore_DP0 pid=40254) INFO 01-25 16:51:27 [kv_cache_utils.py:1310] Maximum concurrency for 2,048 tokens per request: 61.47x
(EngineCore_DP0 pid=40254) INFO 01-25 16:51:27 [core.py:273] init engine (p

In [10]:
ALIAS_PROMPT = """You are an expert entity normalizer.

Your task is to generate alias surface forms for a given canonical concept title.
These aliases are used to map different user-written strings to the SAME concept node
in a concept graph.

Return ONLY valid, raw JSON. No extra text. No markdown.

### Rules:
- "alias" MUST be a list of UNIQUE strings (no duplicates).
- Do NOT include the canonical title itself.
- Do NOT include trivial variants that are the same as the canonical title
  (case-only changes, extra spaces, or punctuation-only changes).
- If you cannot find at least 1 safe alias, output: {"alias": []}
- Provide 1 to 6 aliases max.

### What counts as an alias
Generate only realistic surface-form variants that people actually write:
- Case variants (upper/lower/mixed)
- Spacing variants (with or without spaces)
- Punctuation variants (hyphens, dots, slashes)
- Acronyms OR full names ONLY if they are unambiguous and commonly used
- Very common typos or misspellings (max 2–3)

Do NOT be creative. Be conservative.

### Examples (for pattern understanding only — do NOT copy)

Input: "machine learning"
Output: {"alias": ["ML", "MachineLearning", "machine-learning"]}

Input: "Wi-Fi"
Output: {"alias": ["wifi", "WiFi", "wi fi"]}

Input: "Seq2Seq"
Output: {"alias": ["seq2seq", "Seq2Seq", "sequence-to-sequence", "sequence to sequence"]}

### Task
Canonical Title: "{INPUT_TITLE}"
Output: """

def add_alias_column(node_df: pd.DataFrame) -> pd.DataFrame:
    df = node_df.copy()
    titles = df["name"].dropna().astype(str).unique().tolist()

    chat_inputs = []
    for title in tqdm(titles, desc="alias: build prompts"):
        system_msg = ALIAS_PROMPT.replace("{INPUT_TITLE}", title)
        user_msg = 'Return ONLY raw JSON in the form {"alias": [...]}'
        chat_inputs.append(build_chat_prompt(system_msg, user_msg))

    generations = llm.generate(chat_inputs, sampling)

    alias_map = {}
    raw_map = {}
    ok_map = {}

    for title, gen in tqdm(list(zip(titles, generations)), total=len(titles), desc="alias: parse outputs"):
        raw = gen.outputs[0].text.strip()
        raw_map[title] = raw

        parsed = extract_json(raw)
        ok = isinstance(parsed, dict) and isinstance(parsed.get("alias"), list)
        ok_map[title] = ok

        cleaned = []
        if ok:
            dedup = set()
            for item in parsed["alias"]:
                if not isinstance(item, str):
                    continue
                s = item.strip()
                if not s:
                    continue
                if s == title:   # canonical title 제외
                    continue
                if s not in dedup:
                    dedup.add(s)
                    cleaned.append(s)

        alias_map[title] = cleaned

    df["alias"] = df["name"].astype(str).map(alias_map)
    df["alias_raw"] = df["name"].astype(str).map(raw_map)
    df["alias_ok"] = df["name"].astype(str).map(ok_map)
    return df


In [11]:
preprocess_nodes = add_alias_column(nodes)

alias: parse outputs: 100%|██████████| 182/182 [00:00<00:00, 78949.56it/s]


In [12]:
preprocess_nodes

,id,name,tb_name,link,alias,alias_raw,alias_ok
0,2,Anaphora_(linguistics),Anaphora Resolution,https://en.wikipedia.org/wiki/Anaphora_(lingui...,"[anaphora, anaphora linguistics, anaphora in l...","{""alias"": [""anaphora"", ""anaphora linguistics"",...",True
1,3,Generative_adversarial_network,Generative Adversarial Networks,https://en.wikipedia.org/wiki/Generative_adver...,"[generative adversarial network, GAN, generati...","{""alias"": [""generative adversarial network"", ""...",True
2,4,Neural_machine_translation,Neural Machine Translation,https://en.wikipedia.org/wiki/Neural_machine_t...,"[neural machine translation, neural_machine_tr...","{""alias"": [""neural machine translation"", ""neur...",True
3,5,Autoencoder,Variational Autoencoders,https://en.wikipedia.org/wiki/Autoencoder#Vari...,"[auto encoder, autoencoder, AutoEncoder, auto-...","{""alias"": [""auto encoder"", ""autoencoder"", ""Aut...",True
4,6,IBM_alignment_models,IBM Translation Models,https://en.wikipedia.org/wiki/IBM_alignment_mo...,"[ibm alignment models, IBM alignment models, i...","{""alias"": [""ibm alignment models"", ""IBM alignm...",True
...,...,...,...,...,...,...,...
190,192,Greedy_algorithm,Greedy algorithms,https://en.wikipedia.org/wiki/Greedy_algorithm,"[greedy algorithm, greedy_algorithm, greedy-al...","{""alias"": [""greedy algorithm"", ""greedy_algorit...",True
194,196,Artificial_intelligence_(video_games),Game playing in AI,https://en.wikipedia.org/wiki/Artificial_intel...,"[artificial intelligence in video games, AI in...","{""alias"": [""artificial intelligence in video g...",True
196,198,Lambda_calculus,Lambda calculus,https://en.wikipedia.org/wiki/Lambda_calculus,"[lambda calculus, Lambda calculus, lambda-calc...","{""alias"": [""lambda calculus"", ""Lambda calculus...",True
197,199,Highway_network,Highway networks,https://en.wikipedia.org/wiki/Highway_network,"[highway network, highway_network, highwaynetw...","{""alias"": [""highway network"", ""highway_network...",True


In [13]:
print(preprocess_nodes["alias_ok"].value_counts(dropna=False))
preprocess_nodes.loc[~preprocess_nodes["alias_ok"], ["name", "alias", "alias_raw"]].head(10)

alias_ok
True    184
Name: count, dtype: int64


,name,alias,alias_raw


In [15]:
EDGE_PROMPT = """You are an expert curriculum designer and concept dependency evaluator.

Your task is to judge whether PREREQ is prerequisite knowledge for understanding TARGET.

Return ONLY valid JSON.
No explanations.
No markdown.
No extra text.

The output JSON must have exactly these fields:
{
  "strength": number (float between 0 and 1),
  "reason": string (1–2 short sentences, generic, no names or dates)
}

strength guidelines:
- 0.9–1.0: required to understand TARGET
- 0.6–0.8: very helpful but not required
- 0.3–0.5: weak background
- 0.0–0.2: not a prerequisite

If PREREQ is only loosely related, use strength <= 0.2.
If PREREQ is a subtype or example of TARGET, use strength <= 0.2.
If PREREQ is a foundational concept used to define or derive TARGET, use strength >= 0.6.

Example:
TARGET: Backpropagation
PREREQ: Chain Rule
Output:
{"strength":0.95,"reason":"Backpropagation relies on the chain rule to compute gradients through composed functions."}

Now evaluate:

TARGET: {TARGET}
PREREQ: {PREREQ}

Output:
"""

def add_edge_reason_strength(edge_df: pd.DataFrame) -> pd.DataFrame:
    df = edge_df.copy()
    key_df = df[["name", "prereq_name"]].dropna().drop_duplicates().astype(str)
    key_pairs = list(key_df.itertuples(index=False, name=None))

    chat_inputs = []
    for target, prereq in tqdm(key_pairs, desc="edge: build prompts"):
        system_msg = EDGE_PROMPT.replace("{TARGET}", target).replace("{PREREQ}", prereq)
        user_msg = "Return raw JSON only."
        chat_inputs.append(build_chat_prompt(system_msg, user_msg))

    generations = llm.generate(chat_inputs, sampling)

    # (target, prereq) -> (strength, reason, raw, ok)
    score_map: dict[tuple[str, str], tuple[object, object, str, bool]] = {}

    for (target, prereq), gen in tqdm(
        list(zip(key_pairs, generations)),
        total=len(key_pairs),
        desc="edge: parse outputs",
    ):
        raw = gen.outputs[0].text.strip()
        parsed = extract_json(raw)

        ok = isinstance(parsed, dict) and ("strength" in parsed or "reason" in parsed)

        strength = pd.NA
        reason = pd.NA

        if isinstance(parsed, dict):
            s = parsed.get("strength")
            r = parsed.get("reason")

            if isinstance(s, (int, float)):
                strength = float(max(0.0, min(1.0, s)))
            if isinstance(r, str) and r.strip():
                reason = r.strip()

        score_map[(target, prereq)] = (strength, reason, raw, ok)

    def _get(row, idx):
        return score_map.get((str(row["name"]), str(row["prereq_name"])),
                             (pd.NA, pd.NA, "", False))[idx]

    df["strength"] = df.apply(lambda row: _get(row, 0), axis=1)
    df["reason"]   = df.apply(lambda row: _get(row, 1), axis=1)
    df["edge_raw"] = df.apply(lambda row: _get(row, 2), axis=1)
    df["edge_ok"]  = df.apply(lambda row: _get(row, 3), axis=1)

    return df


In [16]:
preprocess_edges = add_edge_reason_strength(edges)

edge: parse outputs: 100%|██████████| 572/572 [00:00<00:00, 105101.06it/s]


In [17]:
preprocess_edges

,name,prereq_name,strength,reason,edge_raw,edge_ok
48,Word-sense_disambiguation,Anaphora_(linguistics),0.30,"Anaphora relates to pronoun resolution, which ...","{""strength"":0.3,""reason"":""Anaphora relates to ...",True
68,Dialog_system,Anaphora_(linguistics),0.70,Anaphora helps in understanding how references...,"{""strength"":0.7,""reason"":""Anaphora helps in un...",True
207,Autoencoder,Generative_adversarial_network,0.20,Generative adversarial networks and autoencode...,"{""strength"":0.2,""reason"":""Generative adversari...",True
436,Semantic_analysis_(computational),Neural_machine_translation,0.70,Neural machine translation benefits from seman...,"{""strength"":0.7,""reason"":""Neural machine trans...",True
558,Statistical_machine_translation,Neural_machine_translation,0.85,Neural machine translation builds on statistic...,"{""strength"":0.85,""reason"":""Neural machine tran...",True
...,...,...,...,...,...,...
39018,Chatbot,Greedy_algorithm,0.10,Greedy algorithms are a specific optimization ...,"{""strength"":0.1,""reason"":""Greedy algorithms ar...",True
39073,Beam_search,Greedy_algorithm,0.85,Beam search uses a greedy approach to maintain...,"{""strength"":0.85,""reason"":""Beam search uses a ...",True
40290,Bootstrap_aggregating,Lambda_calculus,0.10,Lambda calculus is unrelated to bootstrap aggr...,"{""strength"":0.1,""reason"":""Lambda calculus is u...",True
40296,Propositional_calculus,Lambda_calculus,0.20,Lambda calculus and propositional calculus are...,"{""strength"":0.2,""reason"":""Lambda calculus and ...",True


In [18]:
preprocess_edges.iloc[1]['reason']

'Anaphora helps in understanding how references are resolved in conversations, which is important for dialog systems to maintain coherence.'

In [19]:
print(preprocess_edges["edge_ok"].value_counts(dropna=False))
preprocess_edges.loc[~preprocess_edges["edge_ok"], ["name", "edge_raw"]].head(10)

edge_ok
True    577
Name: count, dtype: int64


,name,edge_raw


In [20]:
preprocess_edges['strength'].describe()

count    577.000000
mean       0.585182
std        0.268357
min        0.000000
25%        0.400000
50%        0.700000
75%        0.850000
max        0.950000
Name: strength, dtype: float64

### Node 용 데이터셋 저장

In [21]:
## name, link, alias

preprocess_nodes[['name', 'link', 'alias']].to_csv('../../data/kc_dataset/tutorialbank/tutorialbank_node.csv', index=False)

### Edge 용 데이터셋 저장

In [ ]:
## name, prereq_name, strength, reason

preprocess_edges[['name', 'prereq_name', 'strength', 'reason']].to_csv('../../data/kc_dataset/tutorialbank/tutorialbank_edge.csv', index=False)